In [ ]:
!pip install datasets transformers


In [ ]:
!pip install numpy==1.26.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 23.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


NumPy version: 2.0.2


In [ ]:
import pandas as pd
import torch
import time
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# -----------------------
# 1. Load Dataset
# -----------------------
df = pd.read_csv("/content/balanced_dataset2.csv")
label_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
id2label = {v: k for k, v in label_map.items()}
df['label'] = df['Sentiment'].map(label_map)

In [ ]:
# -----------------------
# 2. Stratified Split
# -----------------------
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['Text'].tolist(), df['label'].tolist(),
    test_size=0.2, random_state=42, stratify=df['label']
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.5, random_state=42, stratify=temp_labels
)

In [ ]:
# -----------------------
# 3. Tokenization
# -----------------------
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels}).map(tokenize, batched=True)
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels}).map(tokenize, batched=True)
test_dataset = Dataset.from_dict({'text': test_texts, 'label': test_labels}).map(tokenize, batched=True)

for dataset in [train_dataset, val_dataset, test_dataset]:
    dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

Map:   0%|          | 0/25336 [00:00<?, ? examples/s]

Map:   0%|          | 0/3167 [00:00<?, ? examples/s]

Map:   0%|          | 0/3168 [00:00<?, ? examples/s]

In [10]:
# -----------------------
# 4. Model & Training Setup
# -----------------------
model = XLMRobertaForSequenceClassification.from_pretrained(
    "xlm-roberta-base", num_labels=3, id2label=id2label, label2id=label_map
)

training_args = TrainingArguments(
    output_dir="./xlm-roberta-sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# -----------------------
# 5. Metrics
# -----------------------
def compute_metrics(p):
    preds = torch.argmax(torch.tensor(p.predictions), axis=1)
    labels = p.label_ids
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}



In [11]:
# -----------------------
# 6. Trainer
# -----------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)
# -----------------------
# 7. Training with Timing
# -----------------------
print("\n Starting training...")
start_time = time.time()

trainer.train()

end_time = time.time()
elapsed = end_time - start_time
print(f"\n⏱ Total training time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

# -----------------------
# 8. Save Final and Best Model
# -----------------------
trainer.save_model("xlm-roberta-sentiment-final")
tokenizer.save_pretrained("xlm-roberta-sentiment-final")

trainer.model.save_pretrained("xlm-roberta-sentiment-best")
tokenizer.save_pretrained("xlm-roberta-sentiment-best")

print("\n Final model saved to 'xlm-roberta-sentiment-final'")
print(" Best model (based on validation accuracy) saved to 'xlm-roberta-sentiment-best'")


/tmp/ipython-input-11-3584268351.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



 Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.456800,0.372531,0.852542,0.856509,0.852542,0.850635
2,0.274100,0.322149,0.876855,0.878524,0.876855,0.875726
3,0.164800,0.340788,0.880644,0.881937,0.880644,0.879675



⏱ Total training time: 1851.64 seconds (30.86 minutes)

 Final model saved to 'xlm-roberta-sentiment-final'
 Best model (based on validation accuracy) saved to 'xlm-roberta-sentiment-best'


In [ ]:
# -----------------------
# 9. Evaluation (Validation)
# -----------------------
print("\n Validation Evaluation:")
val_preds = trainer.predict(val_dataset)
val_pred_labels = torch.argmax(torch.tensor(val_preds.predictions), axis=1)
val_true_labels = val_preds.label_ids

print(" Validation Accuracy:", accuracy_score(val_true_labels, val_pred_labels))
print(" Validation Report:\n", classification_report(val_true_labels, val_pred_labels, target_names=label_map.keys()))

val_conf_matrix = confusion_matrix(val_true_labels, val_pred_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(val_conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=label_map.keys(), yticklabels=label_map.keys())
plt.title('Confusion Matrix - Validation Set')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# -----------------------
# 10. Evaluation (Test)
# -----------------------
print("\n Test Evaluation:")
test_preds = trainer.predict(test_dataset)
test_pred_labels = torch.argmax(torch.tensor(test_preds.predictions), axis=1)
test_true_labels = test_preds.label_ids

print(" Test Accuracy:", accuracy_score(test_true_labels, test_pred_labels))
print(" Test Report:\n", classification_report(test_true_labels, test_pred_labels, target_names=label_map.keys()))

test_conf_matrix = confusion_matrix(test_true_labels, test_pred_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(test_conf_matrix, annot=True, fmt="d", cmap="Greens", xticklabels=label_map.keys(), yticklabels=label_map.keys())
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
# -----------------------
# 11. Inference on Custom Sentences
# -----------------------
from transformers import pipeline

# Reload the best model and tokenizer
sentiment_pipeline = pipeline(
    "text-classification",
    model="xlm-roberta-sentiment-best",
    tokenizer="xlm-roberta-sentiment-best",
    device=0 if torch.cuda.is_available() else -1
)

# Example Telugu sentences
example_sentences = [
    "ఈ సినిమా చాలా బాగుంది.",  # Positive
    "నాకు ఈ ప్రదర్శన నచ్చలేదు.",  # Negative
    "ఇది సరేలా ఉంది.",  # Neutral
    "వాతావరణం మంచిది కాదు.",  # Negative
    "ఆ వ్యక్తి మాట్లాడిన తీరు చాలా ఆహ్లాదకరంగా ఉంది."  # Positive
]

# Get predictions
predictions = sentiment_pipeline(example_sentences)

# Print results
print("\n💬 Example Predictions:")
for text, pred in zip(example_sentences, predictions):
    print(f"📝 Sentence: {text}")
    print(f"➡ Predicted Sentiment: {pred['label']} (Score: {pred['score']:.4f})\n")

In [ ]:
from transformers import AutoModelForSequenceClassification

# Replace this with your fine-tuned checkpoint
model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-sentiment-final")

# Save the full model to a folder called 'model'
model.save_pretrained("model")  # ✅ this saves the classification head and weights
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-sentiment-final")
tokenizer.save_pretrained("model")


In [ ]:
from transformers import AutoModelForSequenceClassification

# Replace this with your fine-tuned checkpoint
model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-sentiment-best")

# Save the full model to a folder called 'model'
model.save_pretrained("model")  # ✅ this saves the classification head and weights
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-sentiment-best")
tokenizer.save_pretrained("model")


In [19]:
import shutil

shutil.make_archive("xlm-roberta-sentiment-final", 'zip', "xlm-roberta-sentiment-final")


'/content/xlm-roberta-sentiment-final.zip'

In [20]:
from google.colab import files
files.download("xlm-roberta-sentiment-final.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>